In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(
    temperature=0,
    groq_api_key='Your API Key',
    model_name=("llama-3.1-8b-instant")
)

response = llm.invoke("the first person to land on moon was...")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/jobs")
page_data = loader.load().pop().page_content
print(page_data)





















Search Open Nike Jobs










































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu



Selec

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):
        """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

[{'role': 'Principal, Software Engineering, IT',
  'experience': 'Software Engineering',
  'skills': 'IT',
  'description': 'View Job'},
 {'role': 'Senior Manager, Account Engagement',
  'experience': 'Sales & Customer Service',
  'skills': 'Sales & Customer Service',
  'description': 'View Job'},
 {'role': 'Specialist II, Site Experiences LATAM',
  'experience': 'Sales & Customer Service',
  'skills': 'Sales & Customer Service',
  'description': 'View Job'},
 {'role': 'Manager, Software Engineering - Backend, IT',
  'experience': 'Software Engineering',
  'skills': 'IT',
  'description': 'View Job'},
 {'role': 'Retail Associate - NIKE (Delhi)',
  'experience': 'Retail Stores',
  'skills': 'Retail Stores',
  'description': 'View Job'},
 {'role': 'Senior Principal, NVS Pacific',
  'experience': 'Retail Stores',
  'skills': 'Retail Stores',
  'description': 'View Job'},
 {'role': 'Director of Stores, Pacific',
  'experience': 'Retail Stores',
  'skills': 'Retail Stores',
  'description':

In [ ]:
type(json_res)

list

In [ ]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [ ]:
import uuid
import chromadb

client = chromadb.PersistentClient('/content/drive/MyDrive/vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [ ]:
links = collection.query(query_texts=["skills"], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}]]

In [ ]:
job = json_res
job=["skills"]

In [ ]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}

        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools.
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability,
        process optimization, cost reduction, and heightened overall efficiency.
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ.
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):

        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Unlock Efficiency and Scalability with AtliQ

Dear [Client Name],

I came across your company, [Company Name], and was impressed by the innovative work you're doing in [industry/field]. As a business development executive at AtliQ, I'd like to introduce you to our AI & Software Consulting services that can help you streamline your business processes and achieve your goals.

At AtliQ, we specialize in providing tailored solutions that cater to the unique needs of our clients. Our expertise lies in automating tools to facilitate seamless integration of business processes, resulting in scalability, process optimization, cost reduction, and heightened overall efficiency.

I'd like to highlight a few examples of our successful projects that demonstrate our capabilities:

- **Machine Learning with Python**: We've worked on a project with [Client Name] where we implemented a machine learning model using Python to predict customer churn. The results showed a significant reduction in c